# R4 — Joint Multi-Input, Multi-Output (MIMO) Output-Geometry Ablation

Output-geometry ablation only: universe, R1 primary target, raw sequence price representation, normalization axis, and the R1/R3 walk-forward fold schedule (test years 2008-2026) are all held fixed and reused verbatim. Only the output geometry changes -- R3's per-sector scalar-output formulation (`full sequence + target-sector one-hot -> 1 score`) becomes R4's true joint formulation (`full sequence -> 11 simultaneous scores`, one forward pass per week). See `docs/weekly_sector_rotation_r4_mimo_test.md` for the full write-up; this notebook is the executable source of the CSVs it cites. Does not implement R5.

In [1]:
"""
R4 full pipeline (scratchpad validation run). Once verified, this logic is
transcribed cell-by-cell into r4_mimo_ablation.ipynb and executed there --
the notebook is the real executable source of the committed r4_* artifacts.
"""
import json
import time
from math import ceil
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, brier_score_loss, roc_auc_score
from sklearn.preprocessing import StandardScaler

t0 = time.time()

REPO_ROOT = Path("/Users/gavinhussey/Downloads/AtlasQuant")
PRICES_ROOT = REPO_ROOT / "data" / "raw" / "weekly_sector_rotation" / "prices"
FEATURE_PANEL_PATH = REPO_ROOT / "data" / "processed" / "weekly_sector_rotation" / "feature_panel.csv"
OUTPUTS_ROOT = REPO_ROOT / "research" / "strategies" / "weekly_sector_rotation" / "outputs"
R1_TARGETS_PATH = OUTPUTS_ROOT / "r1_target_definitions_full_panel.csv"

SECTOR_ETFS = ["VGT", "VHT", "VCR", "VOX", "VFH", "VIS", "VDC", "VPU", "VAW", "VNQ", "VDE"]
BENCHMARK_ETF = "VTI"

N_VALUES = [4, 13, 26]
RANK_K_VALUES = [2, 3]
MIN_TRAIN_YEARS = 3
BLOCK_SIZE = 8
N_BOOTSTRAP = 5000
BOOTSTRAP_SEED = 20260812
SECTOR_PERM_SEED = 20260813

# ---------------------------------------------------------------------------

## 1. Load panel + R1 targets, weekly sector price matrix, and the corrected R1/R3 walk-forward fold schedule (with the guard assertion against the previously discovered 2007-vs-2008 bug).

In [2]:
# 1. Load panel + R1 targets (identical to R3).
# ---------------------------------------------------------------------------
date_cols_panel = ["date", "week_end", "actual_last_trading_date", "feature_cutoff_timestamp",
                    "next_week_first_trading_date", "next_week_last_trading_date"]
panel = pd.read_csv(FEATURE_PANEL_PATH, parse_dates=date_cols_panel)
panel = panel.sort_values(["symbol", "date"]).reset_index(drop=True)
AUDIT_COLUMNS = ["week_end", "actual_last_trading_date", "week_complete", "feature_cutoff_timestamp",
                  "next_week_first_trading_date", "next_week_last_trading_date"]
FEATURE_COLUMNS = [c for c in panel.columns if c not in {"date", "symbol", *AUDIT_COLUMNS}]

r1_targets = pd.read_csv(R1_TARGETS_PATH, parse_dates=["date"])
panel = panel.merge(
    r1_targets[["date", "symbol", "target_abs_1pct_next_week", "next_week_open_to_close_return",
                "next_week_vti_return", "next_week_excess_return"]],
    on=["date", "symbol"], how="left",
)
panel["year"] = panel["date"].dt.year
PANEL_DATES = sorted(panel["date"].unique())
n_weeks_total = len(PANEL_DATES)

def resample_weekly_close(daily_df: pd.DataFrame) -> pd.Series:
    return daily_df["adjusted_close"].resample("W-FRI").last()

sector_close = {}
for s in SECTOR_ETFS:
    df = pd.read_csv(PRICES_ROOT / f"{s}.csv", parse_dates=["date"]).set_index("date").sort_index()
    sector_close[s] = resample_weekly_close(df)
sector_close_matrix = pd.DataFrame(sector_close)[SECTOR_ETFS].reindex(PANEL_DATES)
assert int(sector_close_matrix.isna().sum().sum()) == 0
SECTOR_MATRIX_VALUES = sector_close_matrix.to_numpy()
assert SECTOR_ETFS == ["VGT", "VHT", "VCR", "VOX", "VFH", "VIS", "VDC", "VPU", "VAW", "VNQ", "VDE"]

_md_for_years = panel.dropna(subset=FEATURE_COLUMNS + ["target_abs_1pct_next_week"])
years_all = sorted(_md_for_years["year"].unique())
test_years = years_all[MIN_TRAIN_YEARS:]
assert test_years == list(range(2008, 2027)), \
    f"fold schedule diverges from validated R1/R3 2008-2026 schedule: {test_years}"

target_lookup = panel.set_index(["date", "symbol"])[
    ["target_abs_1pct_next_week", "next_week_open_to_close_return", "next_week_vti_return", "year"]
]
print(f"[{time.time()-t0:.0f}s] data loaded, {n_weeks_total} weeks, fold schedule verified (2008-2026, 19 folds)")

# ---------------------------------------------------------------------------

[0s] data loaded, 1141 weeks, fold schedule verified (2008-2026, 19 folds)


## 2. MIMO dataset geometry: one sample per week. `X_t` = flatten(N x 11 raw price window), `Y_t` in {0,1}^11 (canonical ticker order), `M_t` = per-sector validity mask (handles the single VDE data-gap week without dropping the other 10 sectors that week).

In [3]:
# 2. MIMO sample builder (one row per week; canonical ticker order everywhere).
# ---------------------------------------------------------------------------
def build_mimo_samples(N: int, matrix: np.ndarray):
    first_usable_idx = N - 1
    rows_meta = []
    X_list, Y_list, M_list, R_list, V_list = [], [], [], [], []
    n_dropped_no_target = 0
    for i in range(first_usable_idx, n_weeks_total):
        date = PANEL_DATES[i]
        window = matrix[i - N + 1: i + 1, :]
        flat = window.flatten()
        labels = np.full(11, np.nan); rets = np.full(11, np.nan); vti_val = np.nan
        for j, sym in enumerate(SECTOR_ETFS):
            row = target_lookup.loc[(date, sym)]
            lbl = row["target_abs_1pct_next_week"]
            if not pd.isna(lbl):
                labels[j] = lbl
            rets[j] = row["next_week_open_to_close_return"]
            if pd.isna(vti_val) and not pd.isna(row["next_week_vti_return"]):
                vti_val = row["next_week_vti_return"]
        mask = ~np.isnan(labels)
        if not mask.any():
            n_dropped_no_target += 1
            continue
        X_list.append(flat); Y_list.append(np.where(mask, labels, 0.0)); M_list.append(mask.astype(np.float64))
        R_list.append(rets); V_list.append(vti_val)
        rows_meta.append({"date": date, "year": pd.Timestamp(date).year})
    meta = pd.DataFrame(rows_meta)
    return (meta, np.vstack(X_list), np.vstack(Y_list), np.vstack(M_list), np.vstack(R_list),
            np.array(V_list), n_dropped_no_target)

samples_by_N = {}
sample_audit_rows = []
for N in N_VALUES:
    meta, X, Y, M, R, V, n_dropped = build_mimo_samples(N, SECTOR_MATRIX_VALUES)
    samples_by_N[N] = {"meta": meta, "X": X, "Y": Y, "M": M, "R": R, "V": V}
    first_usable_idx = N - 1
    first_usable_date = PANEL_DATES[first_usable_idx]
    global_pos_rate = float((Y * M).sum() / M.sum())
    per_sector_rate = {SECTOR_ETFS[j]: float((Y[:, j] * M[:, j]).sum() / M[:, j].sum()) for j in range(11)}
    per_sector_n = {SECTOR_ETFS[j]: int(M[:, j].sum()) for j in range(11)}
    sample_audit_rows.append({
        "N": N, "first_usable_week": str(first_usable_date.date()), "weeks_lost_to_warmup": first_usable_idx,
        "n_weeks_total_samples": len(meta), "n_weeks_dropped_no_target": n_dropped,
        "n_effective_binary_labels": int(M.sum()), "global_positive_rate": global_pos_rate,
        "per_sector_positive_rate_json": json.dumps(per_sector_rate),
        "per_sector_n_labels_json": json.dumps(per_sector_n),
    })
    print(f"[{time.time()-t0:.0f}s] N={N}: {len(meta)} weekly samples (X dim={N*11}), "
          f"{n_dropped} weeks dropped, effective labels={int(M.sum())}, global_pos_rate={global_pos_rate:.4f}")

# per-N per-fold audit (Step 10 granularity)
fold_audit_rows = []
for N in N_VALUES:
    meta = samples_by_N[N]["meta"]; Y = samples_by_N[N]["Y"]; M = samples_by_N[N]["M"]
    years_N = sorted(meta["year"].unique())
    for test_year in test_years:
        train_years_available = [y for y in years_N if y < test_year]
        train_mask = meta["year"].isin(train_years_available).to_numpy()
        test_mask = (meta["year"] == test_year).to_numpy()
        valid = len(train_years_available) >= 2 and train_mask.sum() > 0 and test_mask.sum() > 0
        row = {
            "N": N, "test_year": test_year, "n_train_weeks": int(train_mask.sum()), "n_test_weeks": int(test_mask.sum()),
            "n_train_effective_labels": int(M[train_mask].sum()) if train_mask.sum() else 0,
            "n_test_effective_labels": int(M[test_mask].sum()) if test_mask.sum() else 0,
            "train_positive_rate": float((Y[train_mask] * M[train_mask]).sum() / M[train_mask].sum()) if train_mask.sum() and M[train_mask].sum() else np.nan,
            "test_positive_rate": float((Y[test_mask] * M[test_mask]).sum() / M[test_mask].sum()) if test_mask.sum() and M[test_mask].sum() else np.nan,
            "fold_valid": bool(valid),
        }
        fold_audit_rows.append(row)
r4_sample_audit = pd.DataFrame(fold_audit_rows)
invalid = r4_sample_audit[~r4_sample_audit["fold_valid"]]
print(f"[{time.time()-t0:.0f}s] fold audit built, {len(invalid)} invalid folds across all N (expected 0)")

# ---------------------------------------------------------------------------

[1s] N=4: 1137 weekly samples (X dim=44), 1 weeks dropped, effective labels=12506, global_pos_rate=0.3503


[1s] N=13: 1128 weekly samples (X dim=143), 1 weeks dropped, effective labels=12407, global_pos_rate=0.3490


[1s] N=26: 1115 weekly samples (X dim=286), 1 weeks dropped, effective labels=12264, global_pos_rate=0.3504
[1s] fold audit built, 0 invalid folds across all N (expected 0)


## 3. MIMO-v1 / MIMO-v2 architectures (R1/R3 hidden config, `output_dim=11`) and masked multi-label BCE loss (mean over valid sector x sample elements only), plus the walk-forward evaluator (identical expanding-year schedule, fit/val/test split, and scaler-fit boundary as R3).

In [4]:
# 3. Model definitions: output_dim=11, masked multi-label BCE (mean over valid
#    sector x sample elements). Hidden config identical to R1/R3 NN v1 / NN v2.
# ---------------------------------------------------------------------------
NN_SEED = 20260812
HIDDEN_1, HIDDEN_2 = 16, 8
DROPOUT = 0.3
WEIGHT_DECAY = 1e-4
LEARNING_RATE = 1e-3
MAX_EPOCHS = 300
PATIENCE = 15
OUTPUT_DIM = 11


class SmallMLPMIMO(nn.Module):
    def __init__(self, n_features: int):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_features, HIDDEN_1), nn.ReLU(), nn.Dropout(DROPOUT),
            nn.Linear(HIDDEN_1, HIDDEN_2), nn.ReLU(), nn.Dropout(DROPOUT),
            nn.Linear(HIDDEN_2, OUTPUT_DIM),
        )

    def forward(self, x):
        return self.net(x)


def masked_bce_loss(logits, y, mask):
    elementwise = F.binary_cross_entropy_with_logits(logits, y, reduction="none")
    return (elementwise * mask).sum() / mask.sum().clamp(min=1)


def train_fold_mimo_v1(X_fit, Y_fit, M_fit, X_val, Y_val, M_val, n_features, seed=NN_SEED):
    torch.manual_seed(seed)
    model = SmallMLPMIMO(n_features)
    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    X_fit_t = torch.tensor(X_fit, dtype=torch.float32); Y_fit_t = torch.tensor(Y_fit, dtype=torch.float32)
    M_fit_t = torch.tensor(M_fit, dtype=torch.float32)
    X_val_t = torch.tensor(X_val, dtype=torch.float32); Y_val_t = torch.tensor(Y_val, dtype=torch.float32)
    M_val_t = torch.tensor(M_val, dtype=torch.float32)
    best_val_loss = float("inf"); best_state = None; epochs_without_improvement = 0
    for _ in range(MAX_EPOCHS):
        model.train(); optimizer.zero_grad()
        loss = masked_bce_loss(model(X_fit_t), Y_fit_t, M_fit_t)
        loss.backward(); optimizer.step()
        model.eval()
        with torch.no_grad():
            val_loss = masked_bce_loss(model(X_val_t), Y_val_t, M_val_t).item()
        if val_loss < best_val_loss - 1e-5:
            best_val_loss = val_loss
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1
            if epochs_without_improvement >= PATIENCE:
                break
    model.load_state_dict(best_state); model.eval()
    return model


V2_HIDDEN = 32
V2_DROPOUT = 0.4
V2_WEIGHT_DECAY = 1e-3
V2_LEARNING_RATE = 1e-3
V2_MAX_EPOCHS = 300
V2_PATIENCE = 15
N_SEEDS = 5
SEED_BASE = 20260812


class WideSingleLayerMLPMIMO(nn.Module):
    def __init__(self, n_features: int):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_features, V2_HIDDEN), nn.ReLU(), nn.Dropout(V2_DROPOUT),
            nn.Linear(V2_HIDDEN, OUTPUT_DIM),
        )

    def forward(self, x):
        return self.net(x)


def train_fold_mimo_v2(X_fit, Y_fit, M_fit, X_val, Y_val, M_val, n_features, seed):
    torch.manual_seed(seed)
    model = WideSingleLayerMLPMIMO(n_features)
    optimizer = torch.optim.Adam(model.parameters(), lr=V2_LEARNING_RATE, weight_decay=V2_WEIGHT_DECAY)
    X_fit_t = torch.tensor(X_fit, dtype=torch.float32); Y_fit_t = torch.tensor(Y_fit, dtype=torch.float32)
    M_fit_t = torch.tensor(M_fit, dtype=torch.float32)
    X_val_t = torch.tensor(X_val, dtype=torch.float32); Y_val_t = torch.tensor(Y_val, dtype=torch.float32)
    M_val_t = torch.tensor(M_val, dtype=torch.float32)
    best_val_loss = float("inf"); best_state = None; epochs_without_improvement = 0
    for _ in range(V2_MAX_EPOCHS):
        model.train(); optimizer.zero_grad()
        loss = masked_bce_loss(model(X_fit_t), Y_fit_t, M_fit_t)
        loss.backward(); optimizer.step()
        model.eval()
        with torch.no_grad():
            val_loss = masked_bce_loss(model(X_val_t), Y_val_t, M_val_t).item()
        if val_loss < best_val_loss - 1e-5:
            best_val_loss = val_loss
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1
            if epochs_without_improvement >= V2_PATIENCE:
                break
    model.load_state_dict(best_state); model.eval()
    return model


def make_fold_matrix_mimo(X, idx, scaler=None, fit=False):
    seq = X[idx]
    if fit:
        scaler = StandardScaler().fit(seq)
    return scaler.transform(seq), scaler


def walk_forward_mimo(meta, X, Y, M, R, V, years_, test_years_, n_features, v2=False):
    oof_rows = []
    fold_rows = []
    for test_year in test_years_:
        train_years_available = [y for y in years_ if y < test_year]
        if len(train_years_available) < 2:
            continue
        val_year = train_years_available[-1]
        fit_years = train_years_available[:-1]
        train_mask = meta["year"].isin(train_years_available).to_numpy()
        fit_mask = meta["year"].isin(fit_years).to_numpy()
        val_mask = (meta["year"] == val_year).to_numpy()
        test_mask = (meta["year"] == test_year).to_numpy()
        if train_mask.sum() == 0 or test_mask.sum() == 0 or fit_mask.sum() == 0 or val_mask.sum() == 0:
            continue
        _, scaler = make_fold_matrix_mimo(X, train_mask, fit=True)
        X_fit, _ = make_fold_matrix_mimo(X, fit_mask, scaler=scaler)
        X_val, _ = make_fold_matrix_mimo(X, val_mask, scaler=scaler)
        X_test, _ = make_fold_matrix_mimo(X, test_mask, scaler=scaler)
        Y_fit, M_fit = Y[fit_mask], M[fit_mask]
        Y_val, M_val = Y[val_mask], M[val_mask]
        Y_test, M_test = Y[test_mask], M[test_mask]
        R_test, V_test = R[test_mask], V[test_mask]
        dates_test = meta.loc[test_mask, "date"].to_numpy()

        if v2:
            seed_preds = []
            for so in range(N_SEEDS):
                model = train_fold_mimo_v2(X_fit, Y_fit, M_fit, X_val, Y_val, M_val, n_features, SEED_BASE + so)
                with torch.no_grad():
                    seed_preds.append(torch.sigmoid(model(torch.tensor(X_test, dtype=torch.float32))).numpy())
            preds = np.mean(seed_preds, axis=0)
        else:
            model = train_fold_mimo_v1(X_fit, Y_fit, M_fit, X_val, Y_val, M_val, n_features)
            with torch.no_grad():
                preds = torch.sigmoid(model(torch.tensor(X_test, dtype=torch.float32))).numpy()

        for i_row in range(len(dates_test)):
            for j, sym in enumerate(SECTOR_ETFS):
                if M_test[i_row, j] == 0:
                    continue
                oof_rows.append({
                    "date": dates_test[i_row], "symbol": sym, "predicted_proba": preds[i_row, j],
                    "label_binary": int(Y_test[i_row, j]), "next_week_open_to_close_return": R_test[i_row, j],
                    "next_week_vti_return": V_test[i_row],
                })

        y_flat = Y_test[M_test == 1]; s_flat = preds[M_test == 1]
        fold_auc = roc_auc_score(y_flat, s_flat) if len(np.unique(y_flat)) > 1 else np.nan
        fold_prauc = average_precision_score(y_flat, s_flat) if len(np.unique(y_flat)) > 1 else np.nan
        fold_brier = brier_score_loss(y_flat, s_flat)
        y_fit_flat = Y_fit[M_fit == 1]
        fold_rows.append({
            "test_year": test_year, "n_train_weeks": int(train_mask.sum()), "n_test_weeks": int(test_mask.sum()),
            "n_train_labels": int(M[fit_mask].sum() + M[val_mask].sum()), "n_test_labels": int(M_test.sum()),
            "train_positive_rate": float(np.mean(y_fit_flat)), "test_positive_rate": float(np.mean(y_flat)),
            "fold_auc": fold_auc, "fold_pr_auc": fold_prauc, "fold_brier": fold_brier,
        })
    return pd.DataFrame(oof_rows), pd.DataFrame(fold_rows)


print(f"[{time.time()-t0:.0f}s] Step 3/4 (MIMO model defs + walk-forward evaluator) done.")

# ---------------------------------------------------------------------------

[1s] Step 3/4 (MIMO model defs + walk-forward evaluator) done.


## 4. Top-K metrics and block-bootstrap utilities -- verbatim reuse of R3's implementations.

In [5]:
# 4. Top-K metrics + block bootstrap utilities -- verbatim reuse from R3.
# ---------------------------------------------------------------------------
def topk_metrics_and_series(oof: pd.DataFrame, base_rate: float, k: int):
    weekly_precision, weekly_any_pos, weekly_all_pos = [], [], []
    weekly_mean_ret, weekly_all_sector_mean, weekly_vti_ret, weekly_bottomk_mean = [], [], [], []
    dates_used = []
    for date, g in oof.groupby("date"):
        if len(g) < 11:
            continue
        ranked = g["predicted_proba"].rank(ascending=False, method="first")
        n = len(g)
        top = g.loc[ranked <= k]
        bottom = g.loc[ranked > n - k]
        if top["next_week_open_to_close_return"].isna().any() or top["label_binary"].isna().any():
            continue
        weekly_precision.append(top["label_binary"].mean())
        weekly_any_pos.append(float((top["label_binary"] == 1).any()))
        weekly_all_pos.append(float((top["label_binary"] == 1).all()))
        weekly_mean_ret.append(top["next_week_open_to_close_return"].mean())
        weekly_all_sector_mean.append(g["next_week_open_to_close_return"].mean())
        weekly_vti_ret.append(g["next_week_vti_return"].iloc[0])
        weekly_bottomk_mean.append(bottom["next_week_open_to_close_return"].mean())
        dates_used.append(date)
    series = {
        "date": dates_used, "precision": pd.Series(weekly_precision), "selected_ret": pd.Series(weekly_mean_ret),
        "all_sector_mean": pd.Series(weekly_all_sector_mean), "vti_ret": pd.Series(weekly_vti_ret),
        "bottomk_mean": pd.Series(weekly_bottomk_mean),
    }
    row = {
        "k": k, "n_weeks": len(dates_used), "precision_at_k": series["precision"].mean(),
        "base_positive_rate": base_rate, "precision_minus_base": series["precision"].mean() - base_rate,
        "pct_weeks_at_least_one_positive": pd.Series(weekly_any_pos).mean(),
        "pct_weeks_all_positive": pd.Series(weekly_all_pos).mean(),
        "mean_selected_return": series["selected_ret"].mean(), "median_selected_return": series["selected_ret"].median(),
        "selected_minus_all_sector_mean": (series["selected_ret"] - series["all_sector_mean"]).mean(),
        "selected_minus_vti": (series["selected_ret"] - series["vti_ret"]).mean(),
        "topk_minus_bottomk_spread": (series["selected_ret"] - series["bottomk_mean"]).mean(),
        "win_rate_vs_vti": (series["selected_ret"] > series["vti_ret"]).mean(),
    }
    return row, series


def block_bootstrap_ci(values: pd.Series, block_size: int = BLOCK_SIZE, n_boot: int = N_BOOTSTRAP,
                        seed: int = BOOTSTRAP_SEED) -> dict:
    arr = values.to_numpy()
    n = len(arr)
    rng = np.random.default_rng(seed)
    n_blocks_needed = int(ceil(n / block_size))
    max_start = n - block_size
    boot_means = np.empty(n_boot)
    for b in range(n_boot):
        starts = rng.integers(0, max_start + 1, size=n_blocks_needed)
        sample = np.concatenate([arr[s: s + block_size] for s in starts])[:n]
        boot_means[b] = sample.mean()
    lower, upper = np.percentile(boot_means, [2.5, 97.5])
    return {"point_estimate": arr.mean(), "ci_lower": lower, "ci_upper": upper,
            "n_weeks": n, "significant": bool(lower > 0 or upper < 0)}


def _weekly_yscore_lists(oof: pd.DataFrame):
    dates = sorted(oof["date"].unique())
    y_list, s_list = [], []
    for d in dates:
        g = oof[oof["date"] == d]
        y_list.append(g["label_binary"].to_numpy())
        s_list.append(g["predicted_proba"].to_numpy())
    return dates, y_list, s_list


def block_bootstrap_auc_edge(oof: pd.DataFrame, block_size: int = BLOCK_SIZE, n_boot: int = N_BOOTSTRAP,
                              seed: int = BOOTSTRAP_SEED) -> dict:
    dates, y_list, s_list = _weekly_yscore_lists(oof)
    n = len(dates)
    rng = np.random.default_rng(seed)
    n_blocks_needed = int(ceil(n / block_size))
    max_start = n - block_size
    point_auc = roc_auc_score(oof["label_binary"], oof["predicted_proba"])
    boot_aucs = np.full(n_boot, np.nan)
    for b in range(n_boot):
        starts = rng.integers(0, max_start + 1, size=n_blocks_needed)
        idxs = np.concatenate([np.arange(s, s + block_size) for s in starts])[:n]
        y = np.concatenate([y_list[i] for i in idxs]); s = np.concatenate([s_list[i] for i in idxs])
        if len(np.unique(y)) > 1:
            boot_aucs[b] = roc_auc_score(y, s)
    valid = boot_aucs[~np.isnan(boot_aucs)]
    lower, upper = np.percentile(valid, [2.5, 97.5])
    return {"point_estimate": point_auc - 0.5, "ci_lower": lower - 0.5, "ci_upper": upper - 0.5,
            "n_weeks": n, "significant": bool((lower - 0.5) > 0 or (upper - 0.5) < 0)}


def paired_block_bootstrap_auc_diff(oof_a: pd.DataFrame, oof_b: pd.DataFrame, block_size: int = BLOCK_SIZE,
                                     n_boot: int = N_BOOTSTRAP, seed: int = BOOTSTRAP_SEED) -> dict:
    common_dates = sorted(set(oof_a["date"].unique()) & set(oof_b["date"].unique()))
    oof_a = oof_a[oof_a["date"].isin(common_dates)]
    oof_b = oof_b[oof_b["date"].isin(common_dates)]
    dates_a, y_list_a, s_list_a = _weekly_yscore_lists(oof_a)
    dates_b, y_list_b, s_list_b = _weekly_yscore_lists(oof_b)
    assert dates_a == dates_b, "paired bootstrap requires identical week sets"
    n = len(dates_a)
    rng = np.random.default_rng(seed)
    n_blocks_needed = int(ceil(n / block_size))
    max_start = n - block_size
    point_auc_a = roc_auc_score(oof_a["label_binary"], oof_a["predicted_proba"])
    point_auc_b = roc_auc_score(oof_b["label_binary"], oof_b["predicted_proba"])
    boot_diffs = np.full(n_boot, np.nan)
    for it in range(n_boot):
        starts = rng.integers(0, max_start + 1, size=n_blocks_needed)
        idxs = np.concatenate([np.arange(s, s + block_size) for s in starts])[:n]
        y_a = np.concatenate([y_list_a[i] for i in idxs]); s_a = np.concatenate([s_list_a[i] for i in idxs])
        y_b = np.concatenate([y_list_b[i] for i in idxs]); s_b = np.concatenate([s_list_b[i] for i in idxs])
        if len(np.unique(y_a)) > 1 and len(np.unique(y_b)) > 1:
            boot_diffs[it] = roc_auc_score(y_a, s_a) - roc_auc_score(y_b, s_b)
    valid = boot_diffs[~np.isnan(boot_diffs)]
    lower, upper = np.percentile(valid, [2.5, 97.5])
    point = point_auc_a - point_auc_b
    return {"point_estimate": point, "ci_lower": lower, "ci_upper": upper, "n_weeks": n,
            "significant": bool(lower > 0 or upper < 0)}


def paired_series_diff_bootstrap(oof_a, oof_b, base_rate_a, base_rate_b, k):
    _, series_a = topk_metrics_and_series(oof_a, base_rate_a, k)
    _, series_b = topk_metrics_and_series(oof_b, base_rate_b, k)
    dfa = pd.DataFrame({"date": series_a["date"], "precision": series_a["precision"],
                         "sel_minus_all": series_a["selected_ret"] - series_a["all_sector_mean"],
                         "spread": series_a["selected_ret"] - series_a["bottomk_mean"],
                         "vti_edge": series_a["selected_ret"] - series_a["vti_ret"]}).set_index("date")
    dfb = pd.DataFrame({"date": series_b["date"], "precision": series_b["precision"],
                         "sel_minus_all": series_b["selected_ret"] - series_b["all_sector_mean"],
                         "spread": series_b["selected_ret"] - series_b["bottomk_mean"],
                         "vti_edge": series_b["selected_ret"] - series_b["vti_ret"]}).set_index("date")
    common = dfa.join(dfb, lsuffix="_a", rsuffix="_b", how="inner")
    out = {}
    for qname, col in [("precision_minus_base_diff", "precision"), ("topk_minus_allsector_diff", "sel_minus_all"),
                        ("topk_minus_bottomk_diff", "spread"), ("topk_minus_vti_diff", "vti_edge")]:
        diff_series = common[f"{col}_a"] - common[f"{col}_b"]
        out[qname] = block_bootstrap_ci(diff_series)
    return out


print(f"[{time.time()-t0:.0f}s] Step 4 (topk + bootstrap utilities) defined.")

# ---------------------------------------------------------------------------

[1s] Step 4 (topk + bootstrap utilities) defined.


## 5. Run MIMO-v1 and MIMO-v2 for N in {4, 13, 26} with the full, identity-aligned raw sequence input.

In [6]:
# 5. Run: MIMO-v1 and MIMO-v2, N in {4,13,26}, full identity-aligned input.
# ---------------------------------------------------------------------------
fold_metrics_rows = []
aggregate_rows = []
topk_rows = []
sector_metrics_rows = []
oof_store = {}

for N in N_VALUES:
    meta = samples_by_N[N]["meta"]; X = samples_by_N[N]["X"]; Y = samples_by_N[N]["Y"]
    M = samples_by_N[N]["M"]; R = samples_by_N[N]["R"]; V = samples_by_N[N]["V"]
    years_N = sorted(meta["year"].unique())
    n_features = N * 11

    for model_name, v2 in [("MIMO_v1", False), ("MIMO_v2", True)]:
        oof, folds = walk_forward_mimo(meta, X, Y, M, R, V, years_N, test_years, n_features, v2=v2)
        folds.insert(0, "N", N); folds.insert(1, "model", model_name)
        fold_metrics_rows.append(folds)
        oof_store[(N, model_name)] = oof

        auc = roc_auc_score(oof["label_binary"], oof["predicted_proba"])
        prauc = average_precision_score(oof["label_binary"], oof["predicted_proba"])
        brier = brier_score_loss(oof["label_binary"], oof["predicted_proba"])
        base_rate = oof["label_binary"].mean()
        aggregate_rows.append({
            "N": N, "model": model_name, "n_oof_rows": len(oof), "pooled_auc": auc, "pooled_pr_auc": prauc,
            "pooled_brier": brier, "mean_fold_auc": folds["fold_auc"].mean(), "median_fold_auc": folds["fold_auc"].median(),
            "std_fold_auc": folds["fold_auc"].std(), "mean_fold_pr_auc": folds["fold_pr_auc"].mean(),
            "mean_fold_brier": folds["fold_brier"].mean(), "base_positive_rate": base_rate,
        })

        for k in RANK_K_VALUES:
            row, _ = topk_metrics_and_series(oof, base_rate, k)
            row.update({"N": N, "model": model_name})
            topk_rows.append(row)

        for sym in SECTOR_ETFS:
            sub = oof[oof["symbol"] == sym]
            if len(sub) == 0 or sub["label_binary"].nunique() < 2:
                s_auc, s_prauc = np.nan, np.nan
            else:
                s_auc = roc_auc_score(sub["label_binary"], sub["predicted_proba"])
                s_prauc = average_precision_score(sub["label_binary"], sub["predicted_proba"])
            s_brier = brier_score_loss(sub["label_binary"], sub["predicted_proba"]) if len(sub) else np.nan
            sector_metrics_rows.append({
                "N": N, "model": model_name, "symbol": sym, "n_samples": len(sub),
                "positive_rate": float(sub["label_binary"].mean()) if len(sub) else np.nan,
                "auc": s_auc, "pr_auc": s_prauc, "brier": s_brier,
            })

        print(f"[{time.time()-t0:.0f}s] N={N} {model_name}: pooled_auc={auc:.4f} pr_auc={prauc:.4f} brier={brier:.4f}")

r4_fold_metrics = pd.concat(fold_metrics_rows, ignore_index=True)
r4_aggregate_metrics = pd.DataFrame(aggregate_rows)
r4_topk_all = pd.DataFrame(topk_rows)
r4_sector_metrics = pd.DataFrame(sector_metrics_rows)
print(f"[{time.time()-t0:.0f}s] Step 5 (main MIMO runs) done.")

# ---------------------------------------------------------------------------

[5s] N=4 MIMO_v1: pooled_auc=0.5347 pr_auc=0.3732 brier=0.2318


[9s] N=4 MIMO_v2: pooled_auc=0.5331 pr_auc=0.3719 brier=0.2313


[11s] N=13 MIMO_v1: pooled_auc=0.5221 pr_auc=0.3617 brier=0.2309


[15s] N=13 MIMO_v2: pooled_auc=0.5284 pr_auc=0.3649 brier=0.2334


[17s] N=26 MIMO_v1: pooled_auc=0.5271 pr_auc=0.3652 brier=0.2324


[22s] N=26 MIMO_v2: pooled_auc=0.5240 pr_auc=0.3650 brier=0.2328
[22s] Step 5 (main MIMO runs) done.


## 6. Standalone block-bootstrap CIs (MIMO vs. chance).

In [7]:
# 6. Standalone bootstrap (MIMO vs chance) -- Step 12's baseline CIs.
# ---------------------------------------------------------------------------
bootstrap_rows = []
for N in N_VALUES:
    for model_name in ["MIMO_v1", "MIMO_v2"]:
        oof = oof_store[(N, model_name)]
        base_rate = oof["label_binary"].mean()
        b = block_bootstrap_auc_edge(oof)
        bootstrap_rows.append({"N": N, "model": model_name, "k": np.nan, "quantity": "auc_edge_over_0.5", **b})
        for k in RANK_K_VALUES:
            _, series = topk_metrics_and_series(oof, base_rate, k)
            b = block_bootstrap_ci(series["precision"])
            bootstrap_rows.append({"N": N, "model": model_name, "k": k, "quantity": "precision_minus_base",
                                    "point_estimate": b["point_estimate"] - base_rate,
                                    "ci_lower": b["ci_lower"] - base_rate, "ci_upper": b["ci_upper"] - base_rate,
                                    "n_weeks": b["n_weeks"], "significant": b["significant"]})
            for qname, diff in [
                ("selected_minus_all_sector_mean", series["selected_ret"] - series["all_sector_mean"]),
                ("selected_minus_vti", series["selected_ret"] - series["vti_ret"]),
                ("topk_minus_bottomk_spread", series["selected_ret"] - series["bottomk_mean"]),
            ]:
                b = block_bootstrap_ci(diff)
                bootstrap_rows.append({"N": N, "model": model_name, "k": k, "quantity": qname, **b})
r4_bootstrap_ci = pd.DataFrame(bootstrap_rows)
print(f"[{time.time()-t0:.0f}s] Step 6 (standalone bootstrap) done.")

# ---------------------------------------------------------------------------

[99s] Step 6 (standalone bootstrap) done.


## 7. Rerun R3's scalar-output raw-sequence NN v1 / NN v2 (verbatim R3 code) for N in {4, 13, 26} to obtain row-level OOF predictions for the paired R4-vs-R3 comparison -- R3 itself only persisted aggregate/fold CSVs, not per-row OOF, so this rerun is required to align observations (explicitly permitted by the R4 mandate). Sanity-checked against R3's saved pooled AUCs.

In [8]:
# 7. Rerun R3's scalar raw-sequence NN v1/v2 (LR excluded -- no MIMO-LR control,
#    see Step 9 rationale) for N in {4,13,26} ONLY, to get row-level OOF for the
#    paired R4-vs-R3 comparison (Step 14). R3 itself only persisted aggregate/fold
#    CSVs, not per-row OOF, so this rerun is required to align observations --
#    permitted explicitly by the R4 mandate ("do not rerun unless required to
#    align observations"). Code is R3's own SmallMLP/WideSingleLayerMLP/
#    walk_forward_raw_nn, reproduced verbatim (not modified) to guarantee the
#    comparison is apples-to-apples with the original R3 result.
# ---------------------------------------------------------------------------
class SmallMLP(nn.Module):
    def __init__(self, n_features: int):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_features, HIDDEN_1), nn.ReLU(), nn.Dropout(DROPOUT),
            nn.Linear(HIDDEN_1, HIDDEN_2), nn.ReLU(), nn.Dropout(DROPOUT),
            nn.Linear(HIDDEN_2, 1),
        )

    def forward(self, x):
        return self.net(x).squeeze(-1)


def train_fold_mlp(X_fit, y_fit, X_val, y_val, n_features, seed=NN_SEED):
    torch.manual_seed(seed)
    model = SmallMLP(n_features)
    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    loss_fn = nn.BCEWithLogitsLoss()
    X_fit_t = torch.tensor(X_fit, dtype=torch.float32); y_fit_t = torch.tensor(y_fit, dtype=torch.float32)
    X_val_t = torch.tensor(X_val, dtype=torch.float32); y_val_t = torch.tensor(y_val, dtype=torch.float32)
    best_val_loss = float("inf"); best_state = None; epochs_without_improvement = 0
    for _ in range(MAX_EPOCHS):
        model.train(); optimizer.zero_grad()
        loss = loss_fn(model(X_fit_t), y_fit_t)
        loss.backward(); optimizer.step()
        model.eval()
        with torch.no_grad():
            val_loss = loss_fn(model(X_val_t), y_val_t).item()
        if val_loss < best_val_loss - 1e-5:
            best_val_loss = val_loss
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1
            if epochs_without_improvement >= PATIENCE:
                break
    model.load_state_dict(best_state); model.eval()
    return model


class WideSingleLayerMLP(nn.Module):
    def __init__(self, n_features: int):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_features, V2_HIDDEN), nn.ReLU(), nn.Dropout(V2_DROPOUT),
            nn.Linear(V2_HIDDEN, 1),
        )

    def forward(self, x):
        return self.net(x).squeeze(-1)


def train_fold_mlp_v2(X_fit, y_fit, X_val, y_val, n_features, seed):
    torch.manual_seed(seed)
    model = WideSingleLayerMLP(n_features)
    optimizer = torch.optim.Adam(model.parameters(), lr=V2_LEARNING_RATE, weight_decay=V2_WEIGHT_DECAY)
    loss_fn = nn.BCEWithLogitsLoss()
    X_fit_t = torch.tensor(X_fit, dtype=torch.float32); y_fit_t = torch.tensor(y_fit, dtype=torch.float32)
    X_val_t = torch.tensor(X_val, dtype=torch.float32); y_val_t = torch.tensor(y_val, dtype=torch.float32)
    best_val_loss = float("inf"); best_state = None; epochs_without_improvement = 0
    for _ in range(V2_MAX_EPOCHS):
        model.train(); optimizer.zero_grad()
        loss = loss_fn(model(X_fit_t), y_fit_t)
        loss.backward(); optimizer.step()
        model.eval()
        with torch.no_grad():
            val_loss = loss_fn(model(X_val_t), y_val_t).item()
        if val_loss < best_val_loss - 1e-5:
            best_val_loss = val_loss
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1
            if epochs_without_improvement >= V2_PATIENCE:
                break
    model.load_state_dict(best_state); model.eval()
    return model


def build_sequence_samples_r3(N: int):
    first_usable_idx = N - 1
    rows_meta = []; X_seq_list = []; onehot_list = []
    for i in range(first_usable_idx, n_weeks_total):
        window = SECTOR_MATRIX_VALUES[i - N + 1: i + 1, :]
        flat = window.flatten()
        date = PANEL_DATES[i]
        for j, sym in enumerate(SECTOR_ETFS):
            key = (date, sym)
            if key not in target_lookup.index:
                continue
            row = target_lookup.loc[key]
            if pd.isna(row["target_abs_1pct_next_week"]):
                continue
            onehot = np.zeros(11, dtype=np.float64); onehot[j] = 1.0
            X_seq_list.append(flat); onehot_list.append(onehot)
            rows_meta.append({"date": date, "symbol": sym, "year": int(row["year"]),
                               "label_binary": int(row["target_abs_1pct_next_week"]),
                               "next_week_open_to_close_return": row["next_week_open_to_close_return"],
                               "next_week_vti_return": row["next_week_vti_return"]})
    meta = pd.DataFrame(rows_meta)
    return meta, np.vstack(X_seq_list), np.vstack(onehot_list)


def make_fold_matrix_r3(X_seq, X_onehot, idx, scaler=None, fit=False):
    seq = X_seq[idx]
    if fit:
        scaler = StandardScaler().fit(seq)
    seq_scaled = scaler.transform(seq)
    return np.hstack([seq_scaled, X_onehot[idx]]), scaler


def walk_forward_raw_nn_r3(meta, X_seq, X_onehot, years_, test_years_, n_features, v2=False):
    oof_frames = []
    for test_year in test_years_:
        train_years_available = [y for y in years_ if y < test_year]
        if len(train_years_available) < 2:
            continue
        val_year = train_years_available[-1]
        fit_years = train_years_available[:-1]
        train_mask = meta["year"].isin(train_years_available).to_numpy()
        fit_mask = meta["year"].isin(fit_years).to_numpy()
        val_mask = (meta["year"] == val_year).to_numpy()
        test_mask = (meta["year"] == test_year).to_numpy()
        if train_mask.sum() == 0 or test_mask.sum() == 0 or fit_mask.sum() == 0 or val_mask.sum() == 0:
            continue
        _, scaler = make_fold_matrix_r3(X_seq, X_onehot, train_mask, fit=True)
        X_fit, _ = make_fold_matrix_r3(X_seq, X_onehot, fit_mask, scaler=scaler)
        X_val, _ = make_fold_matrix_r3(X_seq, X_onehot, val_mask, scaler=scaler)
        X_test, _ = make_fold_matrix_r3(X_seq, X_onehot, test_mask, scaler=scaler)
        y_fit = meta.loc[fit_mask, "label_binary"].to_numpy()
        y_val = meta.loc[val_mask, "label_binary"].to_numpy()

        if v2:
            seed_predictions = []
            for seed_offset in range(N_SEEDS):
                model = train_fold_mlp_v2(X_fit, y_fit, X_val, y_val, n_features, SEED_BASE + seed_offset)
                with torch.no_grad():
                    seed_predictions.append(torch.sigmoid(model(torch.tensor(X_test, dtype=torch.float32))).numpy())
            predicted_proba = np.mean(seed_predictions, axis=0)
        else:
            model = train_fold_mlp(X_fit, y_fit, X_val, y_val, n_features)
            with torch.no_grad():
                predicted_proba = torch.sigmoid(model(torch.tensor(X_test, dtype=torch.float32))).numpy()

        fold = meta.loc[test_mask, ["date", "symbol", "next_week_open_to_close_return",
                                     "next_week_vti_return", "label_binary"]].copy()
        fold["predicted_proba"] = predicted_proba
        oof_frames.append(fold)
    return pd.concat(oof_frames, ignore_index=True)


r3_rerun_oof = {}
r3_sequence_samples = {}
for N in N_VALUES:
    meta_r3, X_seq, X_onehot = build_sequence_samples_r3(N)
    r3_sequence_samples[N] = (meta_r3, X_seq, X_onehot)
    years_N = sorted(meta_r3["year"].unique())
    n_features_r3 = N * 11 + 11
    oof_v1 = walk_forward_raw_nn_r3(meta_r3, X_seq, X_onehot, years_N, test_years, n_features_r3, v2=False)
    oof_v2 = walk_forward_raw_nn_r3(meta_r3, X_seq, X_onehot, years_N, test_years, n_features_r3, v2=True)
    r3_rerun_oof[(N, "NN_v1")] = oof_v1
    r3_rerun_oof[(N, "NN_v2")] = oof_v2
    auc_v1 = roc_auc_score(oof_v1.label_binary, oof_v1.predicted_proba)
    auc_v2 = roc_auc_score(oof_v2.label_binary, oof_v2.predicted_proba)
    print(f"[{time.time()-t0:.0f}s] R3 rerun N={N}: NN_v1 auc={auc_v1:.4f} NN_v2 auc={auc_v2:.4f}")

R3_SAVED_AUC = {  # from committed r3_aggregate_metrics.csv (ordered representation), for sanity-check
    (4, "NN_v1"): 0.5446, (13, "NN_v1"): 0.5345, (26, "NN_v1"): 0.5302,
    (4, "NN_v2"): 0.5374, (13, "NN_v2"): 0.5312, (26, "NN_v2"): 0.5276,
}
for (N, m), saved in R3_SAVED_AUC.items():
    rerun_auc = roc_auc_score(r3_rerun_oof[(N, m)].label_binary, r3_rerun_oof[(N, m)].predicted_proba)
    diff = abs(rerun_auc - saved)
    print(f"  sanity check N={N} {m}: rerun={rerun_auc:.4f} vs saved={saved:.4f} (diff={diff:.4f})")
    assert diff < 0.01, f"R3 rerun diverges materially from saved R3 result for N={N} {m} -- investigate"

print(f"[{time.time()-t0:.0f}s] Step 7 (R3 rerun for pairing, sanity-checked against saved R3 artifacts) done.")

# ---------------------------------------------------------------------------

[105s] R3 rerun N=4: NN_v1 auc=0.5446 NN_v2 auc=0.5374


[115s] R3 rerun N=13: NN_v1 auc=0.5345 NN_v2 auc=0.5312


[126s] R3 rerun N=26: NN_v1 auc=0.5302 NN_v2 auc=0.5276
  sanity check N=4 NN_v1: rerun=0.5446 vs saved=0.5446 (diff=0.0000)
  sanity check N=13 NN_v1: rerun=0.5345 vs saved=0.5345 (diff=0.0000)
  sanity check N=26 NN_v1: rerun=0.5302 vs saved=0.5302 (diff=0.0000)
  sanity check N=4 NN_v2: rerun=0.5374 vs saved=0.5374 (diff=0.0000)
  sanity check N=13 NN_v2: rerun=0.5312 vs saved=0.5312 (diff=0.0000)
  sanity check N=26 NN_v2: rerun=0.5276 vs saved=0.5276 (diff=0.0000)
[126s] Step 7 (R3 rerun for pairing, sanity-checked against saved R3 artifacts) done.


## 8. Paired R4-vs-R3 bootstrap (the primary R4 statistical test) and paired R4-vs-R1-engineered-baseline bootstrap (secondary comparison, using R3's saved R1-baseline OOF).

In [9]:
# 8. Paired R4-vs-R3 bootstrap (Step 14) and R4-vs-R1 bootstrap (Step 15).
# ---------------------------------------------------------------------------
MODEL_PAIR = {"MIMO_v1": "NN_v1", "MIMO_v2": "NN_v2"}
r1_baseline_oof = {}
for name in ["NN_v1", "NN_v2"]:
    r1_baseline_oof[name] = pd.read_csv(OUTPUTS_ROOT / f"r3_r1_baseline_oof_{name}.csv", parse_dates=["date"])

vs_r3_rows = []
vs_r1_rows = []
for N in N_VALUES:
    for mimo_name, scalar_name in MODEL_PAIR.items():
        oof_mimo = oof_store[(N, mimo_name)]
        oof_r3 = r3_rerun_oof[(N, scalar_name)]
        oof_r1 = r1_baseline_oof[scalar_name]
        base_mimo = oof_mimo["label_binary"].mean()
        base_r3 = oof_r3["label_binary"].mean()
        base_r1 = oof_r1["label_binary"].mean()

        auc_diff_r3 = paired_block_bootstrap_auc_diff(oof_mimo, oof_r3)
        vs_r3_rows.append({"N": N, "mimo_model": mimo_name, "scalar_model": scalar_name, "k": np.nan,
                            "quantity": "auc", **auc_diff_r3})
        for k in RANK_K_VALUES:
            diffs = paired_series_diff_bootstrap(oof_mimo, oof_r3, base_mimo, base_r3, k)
            for qname, b in diffs.items():
                vs_r3_rows.append({"N": N, "mimo_model": mimo_name, "scalar_model": scalar_name, "k": k,
                                    "quantity": qname, **b})

        auc_diff_r1 = paired_block_bootstrap_auc_diff(oof_mimo, oof_r1)
        vs_r1_rows.append({"N": N, "mimo_model": mimo_name, "r1_model": scalar_name, "k": np.nan,
                            "quantity": "auc", **auc_diff_r1})
        for k in RANK_K_VALUES:
            diffs = paired_series_diff_bootstrap(oof_mimo, oof_r1, base_mimo, base_r1, k)
            for qname, b in diffs.items():
                vs_r1_rows.append({"N": N, "mimo_model": mimo_name, "r1_model": scalar_name, "k": k,
                                    "quantity": qname, **b})
        print(f"[{time.time()-t0:.0f}s] N={N} {mimo_name}: paired-vs-R3 auc_diff={auc_diff_r3['point_estimate']:.4f} "
              f"(sig={auc_diff_r3['significant']}), paired-vs-R1 auc_diff={auc_diff_r1['point_estimate']:.4f} "
              f"(sig={auc_diff_r1['significant']})")

r4_vs_r3_paired = pd.DataFrame(vs_r3_rows)
r4_vs_r1_paired = pd.DataFrame(vs_r1_rows)
print(f"[{time.time()-t0:.0f}s] Step 8 (paired vs R3 + vs R1 bootstrap) done.")

# ---------------------------------------------------------------------------

[176s] N=4 MIMO_v1: paired-vs-R3 auc_diff=-0.0099 (sig=False), paired-vs-R1 auc_diff=-0.0346 (sig=True)


[225s] N=4 MIMO_v2: paired-vs-R3 auc_diff=-0.0043 (sig=False), paired-vs-R1 auc_diff=-0.0377 (sig=True)


[274s] N=13 MIMO_v1: paired-vs-R3 auc_diff=-0.0124 (sig=False), paired-vs-R1 auc_diff=-0.0473 (sig=True)


[322s] N=13 MIMO_v2: paired-vs-R3 auc_diff=-0.0028 (sig=False), paired-vs-R1 auc_diff=-0.0424 (sig=True)


[370s] N=26 MIMO_v1: paired-vs-R3 auc_diff=-0.0031 (sig=False), paired-vs-R1 auc_diff=-0.0422 (sig=True)


[418s] N=26 MIMO_v2: paired-vs-R3 auc_diff=-0.0035 (sig=False), paired-vs-R1 auc_diff=-0.0468 (sig=True)
[418s] Step 8 (paired vs R3 + vs R1 bootstrap) done.


## 9. Cross-sector learning diagnostic: fixed sector-column identity permutation (predeclared, single diagnostic, not a search).

In [10]:
# 9. Cross-sector diagnostic (Step 16): fixed sector-column permutation. The
#    input columns of the raw window are relabeled under a fixed permutation
#    sigma (so column j holds the price history of sector sigma(j) instead of
#    sector j), for ALL samples train+test, while Y/M/R/V stay tied to the true
#    canonical sector identities and the model's output head ordering. This
#    corrupts the alignment between "own-sector history" and "own-sector output
#    head" that a model exploiting genuine per-sector/cross-sector identity
#    would rely on, without changing the numeric distribution of any column.
# ---------------------------------------------------------------------------
sector_perm = np.random.default_rng(SECTOR_PERM_SEED).permutation(11)
assert not np.array_equal(sector_perm, np.arange(11)), "permutation must actually scramble identities"
print(f"[{time.time()-t0:.0f}s] Step 9: fixed sector permutation sigma={sector_perm.tolist()} "
      f"(seed={SECTOR_PERM_SEED}); column j now holds "
      f"{[SECTOR_ETFS[i] for i in sector_perm]} instead of {SECTOR_ETFS}")

diagnostic_rows = []
oof_diag_store = {}
for N in N_VALUES:
    permuted_matrix = SECTOR_MATRIX_VALUES[:, sector_perm]
    meta_p, X_p, Y_p, M_p, R_p, V_p, _ = build_mimo_samples(N, permuted_matrix)
    # Y/M/R/V must be identical to the true-identity build (labels/returns depend only on
    # symbol+date, not on the price matrix) -- assert to guarantee only X changed.
    meta_true = samples_by_N[N]["meta"]
    assert meta_p["date"].tolist() == meta_true["date"].tolist()
    assert np.array_equal(Y_p, samples_by_N[N]["Y"]) and np.array_equal(M_p, samples_by_N[N]["M"])
    years_N = sorted(meta_p["year"].unique())
    n_features = N * 11
    for model_name, v2 in [("MIMO_v1", False), ("MIMO_v2", True)]:
        oof_p, _ = walk_forward_mimo(meta_p, X_p, Y_p, M_p, R_p, V_p, years_N, test_years, n_features, v2=v2)
        oof_diag_store[(N, model_name)] = oof_p
        oof_full = oof_store[(N, model_name)]
        auc_full = roc_auc_score(oof_full["label_binary"], oof_full["predicted_proba"])
        auc_perm = roc_auc_score(oof_p["label_binary"], oof_p["predicted_proba"])
        base_full = oof_full["label_binary"].mean(); base_perm = oof_p["label_binary"].mean()
        row_full_2, _ = topk_metrics_and_series(oof_full, base_full, 2)
        row_perm_2, _ = topk_metrics_and_series(oof_p, base_perm, 2)
        row_full_3, _ = topk_metrics_and_series(oof_full, base_full, 3)
        row_perm_3, _ = topk_metrics_and_series(oof_p, base_perm, 3)
        diagnostic_rows.append({
            "N": N, "model": model_name, "auc_full_identity": auc_full, "auc_permuted_identity": auc_perm,
            "auc_diff_full_minus_permuted": auc_full - auc_perm,
            "precision_at_2_full": row_full_2["precision_at_k"], "precision_at_2_permuted": row_perm_2["precision_at_k"],
            "precision_at_3_full": row_full_3["precision_at_k"], "precision_at_3_permuted": row_perm_3["precision_at_k"],
            "topk_minus_bottomk_k2_full": row_full_2["topk_minus_bottomk_spread"],
            "topk_minus_bottomk_k2_permuted": row_perm_2["topk_minus_bottomk_spread"],
            "topk_minus_bottomk_k3_full": row_full_3["topk_minus_bottomk_spread"],
            "topk_minus_bottomk_k3_permuted": row_perm_3["topk_minus_bottomk_spread"],
        })
        print(f"[{time.time()-t0:.0f}s] cross-sector diagnostic N={N} {model_name}: "
              f"auc_full={auc_full:.4f} auc_permuted={auc_perm:.4f}")

r4_cross_sector_diagnostic = pd.DataFrame(diagnostic_rows)
r4_cross_sector_diagnostic["sector_permutation"] = str(sector_perm.tolist())
print(f"[{time.time()-t0:.0f}s] Step 9 (cross-sector diagnostic) done.")

# ---------------------------------------------------------------------------

[418s] Step 9: fixed sector permutation sigma=[1, 0, 9, 7, 6, 5, 2, 8, 10, 3, 4] (seed=20260813); column j now holds ['VHT', 'VGT', 'VNQ', 'VPU', 'VDC', 'VIS', 'VCR', 'VAW', 'VDE', 'VOX', 'VFH'] instead of ['VGT', 'VHT', 'VCR', 'VOX', 'VFH', 'VIS', 'VDC', 'VPU', 'VAW', 'VNQ', 'VDE']


[422s] cross-sector diagnostic N=4 MIMO_v1: auc_full=0.5347 auc_permuted=0.5266


[427s] cross-sector diagnostic N=4 MIMO_v2: auc_full=0.5331 auc_permuted=0.5355


[430s] cross-sector diagnostic N=13 MIMO_v1: auc_full=0.5221 auc_permuted=0.5179


[435s] cross-sector diagnostic N=13 MIMO_v2: auc_full=0.5284 auc_permuted=0.5276


[438s] cross-sector diagnostic N=26 MIMO_v1: auc_full=0.5271 auc_permuted=0.5245


[443s] cross-sector diagnostic N=26 MIMO_v2: auc_full=0.5240 auc_permuted=0.5256
[443s] Step 9 (cross-sector diagnostic) done.


## 10. Save all required R4 artifacts.

In [11]:
# 10. Save all artifacts.
# ---------------------------------------------------------------------------
r4_fold_metrics.to_csv(OUTPUTS_ROOT / "r4_fold_metrics.csv", index=False)
r4_aggregate_metrics.to_csv(OUTPUTS_ROOT / "r4_aggregate_metrics.csv", index=False)
r4_sector_metrics.to_csv(OUTPUTS_ROOT / "r4_sector_metrics.csv", index=False)
r4_topk_all[r4_topk_all["k"] == 2].to_csv(OUTPUTS_ROOT / "r4_top2_metrics.csv", index=False)
r4_topk_all[r4_topk_all["k"] == 3].to_csv(OUTPUTS_ROOT / "r4_top3_metrics.csv", index=False)
r4_bootstrap_ci.to_csv(OUTPUTS_ROOT / "r4_bootstrap_ci.csv", index=False)
r4_vs_r3_paired.to_csv(OUTPUTS_ROOT / "r4_vs_r3_paired_comparison.csv", index=False)
r4_vs_r1_paired.to_csv(OUTPUTS_ROOT / "r4_vs_r1_comparison.csv", index=False)
r4_cross_sector_diagnostic.to_csv(OUTPUTS_ROOT / "r4_cross_sector_diagnostic.csv", index=False)
r4_sample_audit.to_csv(OUTPUTS_ROOT / "r4_sample_audit.csv", index=False)
pd.DataFrame(sample_audit_rows).to_csv(OUTPUTS_ROOT / "r4_n_sample_summary.csv", index=False)

r3_scalar_baselines = {
    f"N{N}_{m}_auc": float(roc_auc_score(r3_rerun_oof[(N, m)].label_binary, r3_rerun_oof[(N, m)].predicted_proba))
    for N in N_VALUES for m in ["NN_v1", "NN_v2"]
}
r1_engineered_baselines = {
    f"{m}_full_feature_set_auc": float(roc_auc_score(r1_baseline_oof[m].label_binary, r1_baseline_oof[m].predicted_proba))
    for m in ["NN_v1", "NN_v2"]
}
best_agg = r4_aggregate_metrics
summary = {
    "n_values_tested": N_VALUES,
    "r4_mimo_v1_auc_by_N": {int(N): float(best_agg[(best_agg.N == N) & (best_agg.model == "MIMO_v1")]["pooled_auc"].iloc[0]) for N in N_VALUES},
    "r4_mimo_v2_auc_by_N": {int(N): float(best_agg[(best_agg.N == N) & (best_agg.model == "MIMO_v2")]["pooled_auc"].iloc[0]) for N in N_VALUES},
    "r3_scalar_baselines_rerun": r3_scalar_baselines,
    "r1_engineered_baselines": r1_engineered_baselines,
    "sector_permutation": sector_perm.tolist(),
    "n_dropped_weeks_no_target": {int(N): int(sample_audit_rows[i]["n_weeks_dropped_no_target"]) for i, N in enumerate(N_VALUES)},
    "runtime_seconds": time.time() - t0,
}
with open(OUTPUTS_ROOT / "r4_run_summary.json", "w") as f:
    json.dump(summary, f, indent=2)

print(f"[{time.time()-t0:.0f}s] ALL DONE.")
print(json.dumps(summary, indent=2))

[443s] ALL DONE.
{
  "n_values_tested": [
    4,
    13,
    26
  ],
  "r4_mimo_v1_auc_by_N": {
    "4": 0.5347220503926385,
    "13": 0.5220698752228164,
    "26": 0.5271178879414173
  },
  "r4_mimo_v2_auc_by_N": {
    "4": 0.5331493568434745,
    "13": 0.5284155706508648,
    "26": 0.5240486968251674
  },
  "r3_scalar_baselines_rerun": {
    "N4_NN_v1_auc": 0.5446355831767596,
    "N4_NN_v2_auc": 0.5374043455219927,
    "N13_NN_v1_auc": 0.5344949462831816,
    "N13_NN_v2_auc": 0.5311712097123862,
    "N26_NN_v1_auc": 0.5302481476128534,
    "N26_NN_v2_auc": 0.5275967818085465
  },
  "r1_engineered_baselines": {
    "NN_v1_full_feature_set_auc": 0.5693678469913763,
    "NN_v2_full_feature_set_auc": 0.5708322204557499
  },
  "sector_permutation": [
    1,
    0,
    9,
    7,
    6,
    5,
    2,
    8,
    10,
    3,
    4
  ],
  "n_dropped_weeks_no_target": {
    "4": 1,
    "13": 1,
    "26": 1
  },
  "runtime_seconds": 442.793653011322
}
